In [ ]:
from data import ImagePairDataset
from dataclasses import dataclass
import torch
from torchvision.transforms import v2 as T
from torchvision.transforms.functional import to_pil_image
from typing import Any, Optional, Self

In [ ]:
@dataclass
class Config:

    # Transforms
    ROTATION_JITTER_DEGREES: Optional[float] = None
    TRANSLATE_JITTER: Optional[float] = None
    SCALE_JITTER: Optional[float] = None
    FLIP: bool = False
    BRIGHTNESS_JITTER: Optional[float] = None
    SATURATION_JITTER: Optional[float] = None
    CONTRAST_JITTER: Optional[float] = None
    HUE_JITTER: Optional[float] = None
    BLUR_JITTER: Optional[float] = None
    NOISE_SIGMA: float = 0.0
    
    LEARNING_RATE: float = 0.001

    # TODO ...
    
    @staticmethod
    def generate_config(param_grid: dict[str, Any]) -> Self:
        config = Config()
        # TODO ...
        return config

In [ ]:
def generate_transform(config: Config) -> T.Transform:

    def jitter_to_range(jitter: Optional[float], base: float = 0.0) -> tuple[float, float]:
        if jitter is None:
            return None
        return (base - jitter, base + jitter)

    transforms = []

    transforms.append(T.ToImage())
    transforms.append(T.ToDtype(torch.float32, scale=True))

    transforms.append(T.RandomAffine(degrees=config.ROTATION_JITTER_DEGREES or 0,
                                    translate=jitter_to_range(config.TRANSLATE_JITTER),
                                    scale=jitter_to_range(config.SCALE_JITTER, base=1.0)))

    if config.FLIP == True:
        transforms.append(T.RandomHorizontalFlip())

    transforms.append(T.ColorJitter(brightness=config.BRIGHTNESS_JITTER,
                     contrast=config.CONTRAST_JITTER,
                     saturation=config.SATURATION_JITTER,
                     hue=config.HUE_JITTER))

    if config.BLUR_JITTER is not None:
        transforms.append(T.GaussianBlur(kernel_size=5, sigma=(0.0, config.BLUR_JITTER)))

    if config.NOISE_SIGMA is not None:
        transforms.append(T.GaussianNoise(mean=0, sigma=config.NOISE_SIGMA))

    return T.Compose(transforms)

In [ ]:
config = Config()
transform = generate_transform(config)
data = ImagePairDataset(split='train', transform=transform)

first, second, target = data[0]
display(to_pil_image(first))
display(to_pil_image(second))
print('Equivalent' if target == 1.0 else 'Different')